# Accounting for Survivorship Bias
<!-- ### What is Survivorship Bias?
> Survivorship bias is a common mistake in algorithmic trading where 
only stocks that have **survived** until the current day are considered
when evaluating a strategy.
>
> Survivorship bias therefore ignores stocks that once existed, but no 
longer exist, within a realm of stocks. Stocks may be missed if they 
were:
>* Delisted
>* Acquired
>* Bankrupted
>* Removed due to poor performance 
>
> On the flip side of this, survivors may be represented where they 
shouldn't be. Data may be evaluated on for periods prior to the stocks
existence within a realm of stocks. Stocks may be overrepresented if 
they were:
>* Added late due to increased performance -->
<!-- 
### Why is this a problem?
> Both cases of ignoring removed stocks and overrepresenting later added
stocks mean that the final dataset is not representative of the 
actualised real world scenario. This means that any evaluation cannot be
trusted to be accurate.
>
> Strategies are likely to appears less risky than
they are in reality, due to all companies remaining successful, this can
lead to:
>* Inflated returns
>* Lower observed drawdowns
>* Unrealistically stable performance -->

<!-- ### Notebook
> In this notebook I will focus on the **S&P500 Index**. Stocks are 
constantly added/removed from the index based on perfomance, or outlying
factors such as: mergers, acquisitions, delistings etc.
>
> I will use this to develop a working dataset that will be used 
throughout the trading_algos project that will, where possible:
>* Contain full history of the S&P500 from inception until the current 
date
>* Include all stocks removed from the index prior to the current date,
masking any price data beyond their removal from the index
>* Mask price for any stocks added to the index beyond its inception, up
until the date of addition to the index -->
<!-- 
### Issues
> A comprehensive list of current and historical stocks have been 
retrieved from Wikipedia. Whilst this provides some useful data for the
purpose of this project, it cannot be trusted without fail. -->

In this notebook I will focus on the **S&P500 Index**. Stocks are 
constantly added/removed from the index based on perfomance, or outlying
factors such as: mergers, acquisitions, delistings etc.

I will use this to develop a working dataset that will be used 
throughout the trading_algos project that will, where possible, contain
full history of the S&P500 from inception until the current date, 
**taking survivorship bias into account**.

## What is Survivorship Bias?
Survivorship bias is a common mistake in algorithmic trading where 
only stocks that have **survived** until the current day are considered
when evaluating a strategy.

Survivorship bias therefore ignores stocks that once existed, but no 
longer exist, within a realm of stocks. Stocks may be missed if they 
were:
* Delisted
* Acquired
* Bankrupted
* Removed due to poor performance 

On the flip side of this, survivors may be overrepresented historically. 
Data may be evaluated on for periods prior to the stocks existence 
within a realm of stocks. Stocks may be overrepresented if they were: 
* Added late due to increased performance

## Why is this a problem?
Both cases of ignoring removed stocks and overrepresenting later added
stocks mean that the final dataset is **not representative of the 
actualised real world scenario** that should be evaluated against. 
Therefore any evaluation cannot be trusted to be accurate.

Strategies are likely to **appear less risky than they are in reality**, 
due to all companies remaining successful, this can lead to:
* Inflated returns
* Lower observed drawdowns
* Unrealistically stable performance


## Issues
A comprehensive list of current and historical stocks have been 
retrieved from Wikipedia. Whilst this provides some useful data for the
purpose of this project, it cannot be trusted without fail. Stocks will
be ignored where any error is likely

# Notebook

## Imports

In [1]:
import pandas as pd
import numpy as np

from trading_algos import datasets as tad
from trading_algos.utils import head_tail as ht

%load_ext autoreload
%autoreload 2

2026-04-11 17:36:11.670 | INFO     | trading_algos.config:<module>:11 - PROJ_ROOT path is: /home/jamie/code/JamieW365/trading_algos


## Constants

In [2]:
# Initialize the start and end dates of the S&P500 dataset that we are
# going to create
start_date = '2020-01-01'
end_date = '2025-01-01'

## Load Data

In [3]:
# Load the most recent S&P stock metadata
# get_sp500_meta will return metadata from Wikipedia or a local file 
# source
df_current, df_changes = tad.get_sp500_meta()

In [4]:
# This first table contains information regarding all current components
# of the S&P500, including date of addition.
ht(df_current)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,CIK,Founded
Date added,,,,,,,
1957-03-04,MMM,3M,Industrials,Industrial Conglomerates,"Saint Paul, Minnesota",66740,1902
1957-03-04,ETR,Entergy,Utilities,Electric Utilities,"New Orleans, Louisiana",65984,1913
1957-03-04,PEG,Public Service Enterprise Group,Utilities,Electric Utilities,"Newark, New Jersey",788784,1903
2026-03-23,COHR,Coherent Corp.,Information Technology,Electronic Components,"Saxonburg, Pennsylvania",820318,1971
2026-03-23,LITE,Lumentum,Information Technology,Communications Equipment,"San Jose, California",1633978,2015
2026-04-09,CASY,Casey's,Consumer Staples,Food Retail,"Ankeny, Iowa",726958,1967


In [5]:
# This second table contains information regarding all component changes
# to the S&P500, including date of removal
ht(df_changes)

Effective Date  Added                          Removed                       \
Effective Date Ticker                 Security  Ticker             Security   
1976-07-01        DIS  The Walt Disney Company     AYE     Allegheny Energy   
1976-07-01        BUD           Anheuser Busch     HNG  Houston Natural Gas   
1994-09-30        NCC            National City     MCK             McKesson   
2026-03-23       LITE                 Lumentum     MOH    Molina Healthcare   
2026-03-23        VRT                   Vertiv    MTCH          Match Group   
2026-04-09       CASY                  Casey's    HOLX              Hologic   

Effective Date                                             Reason  
Effective Date                                             Reason  
1976-07-01      Major restructuring of S&P 500 to have fewer i...  
1976-07-01      Major restructuring of S&P 500 to have fewer i...  
1994-09-30      McKesson sold PCS Health Services to Eli Lilly...  
2026-03-23                       Market capitalization change.[7]  
2026-03-23                       Market capitalization change.[7]  
2026-04-09      Blackstone Inc. and TPG Inc. acquired Hologic.[6]

In [6]:
# Some of the current S&P500 will may have been added to the index after
# the start date and so any price data should be nulled prior to
# addition to the index. This frame keeps track of those that have been
# added beyond the start date.

df_added = df_current[(end_date >= df_current.index) & 
                      (df_current.index >= start_date)].copy()
ht(df_added)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,CIK,Founded
Date added,,,,,,,
2020-03-03,IR,Ingersoll Rand,Industrials,Industrial Machinery & Supplies & Components,"Davidson, North Carolina",1699150,1859
2020-04-03,OTIS,Otis Worldwide,Industrials,Industrial Machinery & Supplies & Components,"Farmington, Connecticut",1781335,"2020 (1853, United Technologies spinoff)"
2020-04-03,CARR,Carrier Global,Industrials,Building Products,"Palm Beach Gardens, Florida",1783180,"2020 (1915, United Technologies spinoff)"
2024-12-23,WDAY,"Workday, Inc.",Information Technology,Application Software,"Pleasanton, California",1327811,2005
2024-12-23,LII,Lennox International,Industrials,Building Products,"Richardson, Texas",1069202,1895
2024-12-23,APO,Apollo Global Management,Financials,Asset Management & Custody Banks,"New York City, New York",1858681,1990


In [7]:
# We have a list of stocks that have been removed from the S&P500 at
# some point in time. Any stocks that were removed from the S&P500 after
# our start date should be included until the date of their removal.
df_removed = df_changes['Removed'][(end_date >= df_changes['Removed'].index) &
                                (df_changes['Removed'].index >= start_date)].copy()

# Sometimes spin-off stocks will be temporarily added to the S&P500
# without removing any other stocks. This is to ensure that there are
# no immediate changes in index holding. As a result there may be some
# null entries in the removed table which can be ignored.
df_removed.dropna(inplace=True)
ht(df_removed)

Effective Date,Ticker,Security
2020-01-28,WCG,WellCare
2020-03-02,XEC,Cimarex Energy
2020-04-01,ARNC,Arconic
2024-12-23,CTLT,Catalent
2024-12-23,AMTM,Amentum
2024-12-23,QRVO,Qorvo


In [8]:
# Load all stocks that have been included in the S&P500 at any point
# between our start and end dates
stocks = df_current.Symbol.tolist() + df_removed.Ticker.tolist()
print('Total n =', len(stocks), 'stocks to be loaded')

Total n = 589 stocks to be loaded


## Download Stocks

In [9]:
# Commented out to load data from local source moving forward

# # Load data from Yahoo Finance
# df_stocks = tad.load_data(stocklist,
#                           start_date=start_date,
#                           end_date=end_date)

In [10]:
# Commented out to load data from local source moving forward

# # Some tickers may have been delisted since their into the S&P500. This
# # may be due to a merge or rebrand, in which case data should be 
# # accounted for under it's replacement ticker. These can be removed.
# df_stocks = df_stocks.dropna(axis=1, how='all')

# # Saving this data to avoid future Yahoo Finance calls when rerunning
# tad.save_data(df_stocks,
#               'sap500alltime.csv')

In [11]:
# sap500alltime.csv contains a comprehensive price dataset of ALL stocks
# that have ever been included in the S&P500 index
# Price data for each stock is loaded in it's entirity i.e. containing
# all price data between 1900-01-01 and the most recent download date 
df_stocks = tad.load_data(filename='sap500alltime.csv',
                          tickers=stocks,
                          start_date=start_date,
                          end_date=end_date)

In [12]:
ht(df_stocks)

Price            Close                                                 \
Ticker               A        AAL         AAP        AAPL        ABBV   
Date                                                                    
2020-01-02   82.398026  28.982893  140.549072   72.400513   69.266151   
2020-01-03   81.075073  27.548195  140.557861   71.696625   68.608688   
2020-01-06   81.314751  27.219410  138.247879   72.267944   69.150124   
2024-12-27  134.209549  17.350000   42.825951  254.201370  170.623230   
2024-12-30  133.100540  17.620001   44.760212  250.829803  168.888351   
2024-12-31  133.267197  17.430000   45.965481  249.059448  170.326111   

Price                                                                  ...  \
Ticker            ABNB         ABT       ACGL         ACN        ADBE  ...   
Date                                                                   ...   
2020-01-02         NaN   77.813881  41.268997  192.514999  334.429993  ...   
2020-01-03         NaN   76.865242  41.221451  192.194412  331.809998  ...   
2020-01-06         NaN   77.267960  41.383106  190.939377  333.709991  ...   
2024-12-27  133.384995  112.282875  92.339996  351.166351  446.480011  ...   
2024-12-30  131.809998  110.144440  91.889999  347.528259  445.799988  ...   
2024-12-31  131.410004  110.447128  92.349998  346.838135  444.679993  ...   

Price           Volume                                                        \
Ticker             XOM       XRAY        XRX       XYL        XYZ        YUM   
Date                                                                           
2020-01-02  12456400.0  1556600.0  1581300.0  869500.0  5264700.0  1369900.0   
2020-01-03  17386900.0   910000.0  1042600.0  795100.0  5087100.0  1145500.0   
2020-01-06  20081900.0   751000.0  1199200.0  817300.0  5905200.0  1454100.0   
2024-12-27  11943900.0  1876900.0  1874700.0  552400.0  4140800.0  1146300.0   
2024-12-30  11080800.0  1965500.0  2120300.0  586800.0  5383800.0  1144600.0   
2024-12-31  12387800.0  1595200.0  2035800.0  641600.0  4989400.0  1217100.0   

Price                                                  
Ticker            ZBH      ZBRA       ZION        ZTS  
Date                                                   
2020-01-02  1083972.0  387800.0  1528700.0  1576700.0  
2020-01-03   992405.0  305300.0  1215800.0  1274000.0  
2020-01-06   972423.0  322600.0  1217500.0  2334100.0  
2024-12-27   743400.0  287200.0   923400.0  1800100.0  
2024-12-30  1532000.0  211300.0   827800.0  1531400.0  
2024-12-31   683300.0  327900.0  1426000.0  1327400.0  

[6 rows x 2725 columns]

In [13]:
df_stocks.shape

(1258, 2725)

In [14]:
# Quality check that each level contains data for the same tickers

qual_check = []
prev_col=None
for col1 in df_stocks.columns.get_level_values(0).unique():

    if prev_col:
        qual_check.append((remaining_cols==df_stocks[col1].dropna(axis=1, how='all').columns).all())

    remaining_cols = df_stocks[col1].dropna(axis=1, how='all').columns

    prev_col = col1

if (qual_check == np.True_).all():
    print('Quality check passed')

Quality check passed


## Accounting for Survivorship Bias
Now that we have full price data for all stocks that have ever been a
part of the S&P500, it is important to account for any time period that 
those stocks dropped out or were not included. We know when stocks were 
added or removed from the index, now we need to null price data for the 
periods where it was not included.

### Nulling removed data

In [15]:
df_removed.tail()

Effective Date,Ticker,Security
2024-10-01,BBWI,"Bath & Body Works, Inc."
2024-11-26,MRO,Marathon Oil
2024-12-23,CTLT,Catalent
2024-12-23,AMTM,Amentum
2024-12-23,QRVO,Qorvo


In [16]:
# We know when a stock was removed by the index in the removed dataset

df_removed[df_removed['Ticker']=='QRVO'].index[0]

'2024-12-23'

In [17]:
# Data has been loaded form the start date until the current date

ht(df_stocks.xs('QRVO', axis=1, level=1))

Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,115.959999,117.400002,114.760002,117.260002,1870600.0
2020-01-03,112.339996,115.239998,112.059998,114.059998,2695100.0
2020-01-06,111.639999,112.980003,109.400002,110.830002,3186200.0
2024-12-27,71.419998,71.800003,70.690002,71.790001,2037900.0
2024-12-30,69.730003,70.580002,69.089996,70.580002,2339300.0
2024-12-31,69.930000,70.389999,69.209999,70.000000,2671100.0


In [18]:
# Using the stocks removal date, we can create a proxy index that runs 
# from the start date until the date of stock removal

pd.date_range(start_date, df_removed[df_removed['Ticker']=='QRVO'].index[0])

DatetimeIndex(['2020-01-01', '2020-01-02', '2020-01-03', '2020-01-04',
               '2020-01-05', '2020-01-06', '2020-01-07', '2020-01-08',
               '2020-01-09', '2020-01-10',
               ...
               '2024-12-14', '2024-12-15', '2024-12-16', '2024-12-17',
               '2024-12-18', '2024-12-19', '2024-12-20', '2024-12-21',
               '2024-12-22', '2024-12-23'],
              dtype='datetime64[ns]', length=1819, freq='D')

In [19]:
# Reindexing the removed stock to the new index

df_stocks.loc[:, pd.IndexSlice[: ,'QRVO']] = df_stocks.loc[:, pd.IndexSlice[: ,'QRVO']].\
    reindex(pd.date_range(start_date, df_removed[df_removed['Ticker']=='QRVO'].index[0]))

In [20]:
ht(df_stocks.xs('QRVO', axis=1, level=1))

Price,Close,High,Low,Open,Volume
Date,,,,,
2020-01-02,115.959999,117.400002,114.760002,117.260002,1870600.0
2020-01-03,112.339996,115.239998,112.059998,114.059998,2695100.0
2020-01-06,111.639999,112.980003,109.400002,110.830002,3186200.0
2024-12-27,NaN,NaN,NaN,NaN,NaN
2024-12-30,NaN,NaN,NaN,NaN,NaN
2024-12-31,NaN,NaN,NaN,NaN,NaN


In [21]:
# All price data beyond the removed date has been nulled

ht(df_stocks.loc[:, pd.IndexSlice[: ,'QRVO']].loc[df_removed[df_removed['Ticker']=='QRVO'].index[0]:])

Price,Close,High,Low,Open,Volume
Ticker,QRVO,QRVO,QRVO,QRVO,QRVO
Date,,,,,
2024-12-23,71.540001,72.980003,70.739998,70.82,4348900.0
2024-12-24,NaN,NaN,NaN,NaN,NaN
2024-12-26,NaN,NaN,NaN,NaN,NaN
2024-12-27,NaN,NaN,NaN,NaN,NaN
2024-12-30,NaN,NaN,NaN,NaN,NaN
2024-12-31,NaN,NaN,NaN,NaN,NaN


In [22]:
# All price data leading up to the removed date remains in place

ht(df_stocks.loc[:, pd.IndexSlice[: ,'QRVO']].loc[:df_removed[df_removed['Ticker']=='QRVO'].index[0]].tail())

Price,Close,High,Low,Open,Volume
Ticker,QRVO,QRVO,QRVO,QRVO,QRVO
Date,,,,,
2024-12-17,70.949997,71.320000,69.820000,70.269997,2823700.0
2024-12-18,68.500000,71.320000,68.095001,71.050003,3854400.0
2024-12-19,68.800003,69.730003,68.400002,68.690002,4105200.0
2024-12-19,68.800003,69.730003,68.400002,68.690002,4105200.0
2024-12-20,70.849998,71.320000,68.180000,68.910004,30048600.0
2024-12-23,71.540001,72.980003,70.739998,70.820000,4348900.0


### Nulling Added Data

In [23]:
df_added.loc[:end_date].tail()

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,CIK,Founded
Date added,,,,,,,
2024-09-23,ERIE,Erie Indemnity,Financials,Insurance Brokers,"Erie, Pennsylvania",922621,1925
2024-11-26,TPL,Texas Pacific Land Corporation,Energy,Oil & Gas Exploration & Production,"Dallas, Texas",1811074,1888
2024-12-23,WDAY,"Workday, Inc.",Information Technology,Application Software,"Pleasanton, California",1327811,2005
2024-12-23,LII,Lennox International,Industrials,Building Products,"Richardson, Texas",1069202,1895
2024-12-23,APO,Apollo Global Management,Financials,Asset Management & Custody Banks,"New York City, New York",1858681,1990


In [24]:
# We know when a stock was added by the index in the added dataset

df_added[df_added['Symbol']=='WDAY'].index[0]

'2024-12-23'

In [25]:
# Data has been downloaded from the start date, and not the date that 
# the stock was added to the S&P500

ht(df_stocks.loc[:, pd.IndexSlice[: ,'WDAY']])

Price,Close,High,Low,Open,Volume
Ticker,WDAY,WDAY,WDAY,WDAY,WDAY
Date,,,,,
2020-01-02,167.460007,168.720001,165.710007,166.100006,1503000.0
2020-01-03,168.440002,168.860001,164.960007,165.000000,1276300.0
2020-01-06,169.490005,170.440002,166.350006,166.990005,1623300.0
2024-12-27,266.239990,268.359985,263.269989,267.600006,1602800.0
2024-12-30,262.000000,264.519989,259.329987,263.570007,1755600.0
2024-12-31,258.029999,263.339996,256.190002,262.239990,1587700.0


In [26]:
# Creating a new index that runs from the given date of stock addition
# until the date of chosen end date

pd.date_range(df_added[df_added['Symbol']=='WDAY'].index[0], end_date)

DatetimeIndex(['2024-12-23', '2024-12-24', '2024-12-25', '2024-12-26',
               '2024-12-27', '2024-12-28', '2024-12-29', '2024-12-30',
               '2024-12-31', '2025-01-01'],
              dtype='datetime64[ns]', freq='D')

In [27]:
# Using the stocks addition date, we can create a proxy index that runs 
# from the date of addition until the end date

df_stocks.loc[:, pd.IndexSlice[: ,'WDAY']] = df_stocks.loc[:, pd.IndexSlice[: ,'WDAY']].\
    reindex(pd.date_range(df_added[df_added['Symbol']=='WDAY'].index[0], end_date))

In [28]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'WDAY']])

Price,Close,High,Low,Open,Volume
Ticker,WDAY,WDAY,WDAY,WDAY,WDAY
Date,,,,,
2020-01-02,NaN,NaN,NaN,NaN,NaN
2020-01-03,NaN,NaN,NaN,NaN,NaN
2020-01-06,NaN,NaN,NaN,NaN,NaN
2024-12-27,266.239990,268.359985,263.269989,267.600006,1602800.0
2024-12-30,262.000000,264.519989,259.329987,263.570007,1755600.0
2024-12-31,258.029999,263.339996,256.190002,262.239990,1587700.0


In [29]:
# All price data before the added date has been nulled

ht(df_stocks.loc[:, pd.IndexSlice[: ,'WDAY']].loc[:df_added[df_added['Symbol']=='WDAY'].index[0]].tail())

Price,Close,High,Low,Open,Volume
Ticker,WDAY,WDAY,WDAY,WDAY,WDAY
Date,,,,,
2024-12-17,NaN,NaN,NaN,NaN,NaN
2024-12-18,NaN,NaN,NaN,NaN,NaN
2024-12-19,NaN,NaN,NaN,NaN,NaN
2024-12-19,NaN,NaN,NaN,NaN,NaN
2024-12-20,NaN,NaN,NaN,NaN,NaN
2024-12-23,265.390015,272.0,263.859985,270.109985,3676000.0


In [30]:
# All price data after the added date remains

ht(df_stocks.loc[:, pd.IndexSlice[: ,'WDAY']].loc[df_added[df_added['Symbol']=='WDAY'].index[0]:].head())

Price,Close,High,Low,Open,Volume
Ticker,WDAY,WDAY,WDAY,WDAY,WDAY
Date,,,,,
2024-12-23,265.390015,272.000000,263.859985,270.109985,3676000.0
2024-12-24,269.040009,269.089996,264.750000,266.299988,850500.0
2024-12-26,269.380005,270.140015,265.500000,266.399994,1243900.0
2024-12-26,269.380005,270.140015,265.500000,266.399994,1243900.0
2024-12-27,266.239990,268.359985,263.269989,267.600006,1602800.0
2024-12-30,262.000000,264.519989,259.329987,263.570007,1755600.0


### Creating functions to apply nulling filters to all stocks

In [31]:
def filter_removed(ticker):
    # This will reindex any removed stocks from the chosen start date
    # until the date of stock removal
    
    if ticker in df_stocks.columns.get_level_values(1).unique():
        df_stocks.loc[:, pd.IndexSlice[: ,ticker]] =\
              df_stocks.loc[:, pd.IndexSlice[: ,ticker]]\
                .reindex(pd.date_range(start_date, 
                                       df_removed[df_removed['Ticker']==ticker].index[0],
                                         freq='B'
                                      )
                        )

def filter_added(ticker):
    # This will reindex any added stocks from the date of stock addition
    # until the chosen end date

    if ticker in df_stocks.columns.get_level_values(1).unique():
        df_stocks.loc[:, pd.IndexSlice[: ,ticker]] =\
              df_stocks.loc[:, pd.IndexSlice[: ,ticker]]\
                .reindex(pd.date_range(df_added[df_added['Symbol']==ticker].index[0], 
                                       end_date, 
                                       freq='B'
                                      )
                        )

In [32]:
# Reindex all removed stocks

for ticker in df_removed['Ticker']:
    filter_removed(ticker)

In [33]:
# Reindex all added stocks

for ticker in df_added['Symbol']:
    filter_added(ticker)

### Quality Check

#### Removed

In [34]:
df_removed[df_removed['Ticker'].isin(df_stocks.columns.get_level_values(1).unique())].sample(3)

Effective Date,Ticker,Security
2021-09-20,NOV,Nov
2023-10-03,DXC,DXC Technology
2024-10-01,BBWI,"Bath & Body Works, Inc."


In [35]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'XRX']].loc[df_removed[df_removed['Ticker']=='XRX'].index[0]:],2)

Price,Close,High,Low,Open,Volume
Ticker,XRX,XRX,XRX,XRX,XRX
Date,,,,,
2021-03-22,18.333574,18.636181,18.104774,18.591898,2760300.0
2021-03-23,NaN,NaN,NaN,NaN,NaN
2024-12-30,NaN,NaN,NaN,NaN,NaN
2024-12-31,NaN,NaN,NaN,NaN,NaN


In [36]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'PENN']].loc[df_removed[df_removed['Ticker']=='PENN'].index[0]:],2)

Price,Close,High,Low,Open,Volume
Ticker,PENN,PENN,PENN,PENN,PENN
Date,,,,,
2022-09-19,31.41,31.67,30.01,30.01,7171400.0
2022-09-20,NaN,NaN,NaN,NaN,NaN
2024-12-30,NaN,NaN,NaN,NaN,NaN
2024-12-31,NaN,NaN,NaN,NaN,NaN


In [37]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'VFC']].loc[df_removed[df_removed['Ticker']=='VFC'].index[0]:],2)

Price,Close,High,Low,Open,Volume
Ticker,VFC,VFC,VFC,VFC,VFC
Date,,,,,
2024-04-03,13.314236,13.706675,13.22809,13.601387,8129900.0
2024-04-04,NaN,NaN,NaN,NaN,NaN
2024-12-30,NaN,NaN,NaN,NaN,NaN
2024-12-31,NaN,NaN,NaN,NaN,NaN


#### Added

In [38]:
df_added[df_added['Symbol'].isin(df_stocks.columns.get_level_values(1).unique())].sample(3)

,Symbol,Security,GICS Sector,GICS Sub-Industry,Headquarters Location,CIK,Founded
Date added,,,,,,,
2024-06-24,KKR,KKR & Co.,Financials,Asset Management & Custody Banks,"New York City, New York",1404912,1976
2024-06-24,CRWD,CrowdStrike,Information Technology,Systems Software,"Austin, Texas",1535527,2011
2020-04-03,CARR,Carrier Global,Industrials,Building Products,"Palm Beach Gardens, Florida",1783180,"2020 (1915, United Technologies spinoff)"


In [39]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'POOL']].loc[:df_added[df_added['Symbol']=='POOL'].index[0]].tail(),2)

Price,Close,High,Low,Open,Volume
Ticker,POOL,POOL,POOL,POOL,POOL
Date,,,,,
2020-10-01,NaN,NaN,NaN,NaN,NaN
2020-10-02,NaN,NaN,NaN,NaN,NaN
2020-10-06,NaN,NaN,NaN,NaN,NaN
2020-10-07,310.422638,312.631638,306.781495,307.998314,547600.0


In [40]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'ACGL']].loc[:df_added[df_added['Symbol']=='ACGL'].index[0]].tail(),2)

Price,Close,High,Low,Open,Volume
Ticker,ACGL,ACGL,ACGL,ACGL,ACGL
Date,,,,,
2022-10-26,NaN,NaN,NaN,NaN,NaN
2022-10-27,NaN,NaN,NaN,NaN,NaN
2022-10-31,NaN,NaN,NaN,NaN,NaN
2022-11-01,52.717812,55.104569,52.613216,54.695682,9579300.0


In [41]:
ht(df_stocks.loc[:, pd.IndexSlice[: ,'TRMB']].loc[:df_added[df_added['Symbol']=='TRMB'].index[0]].tail(),2)

Price,Close,High,Low,Open,Volume
Ticker,TRMB,TRMB,TRMB,TRMB,TRMB
Date,,,,,
2021-01-14,NaN,NaN,NaN,NaN,NaN
2021-01-15,NaN,NaN,NaN,NaN,NaN
2021-01-20,NaN,NaN,NaN,NaN,NaN
2021-01-21,70.309998,72.800003,69.709999,72.800003,2860600.0


## Generating a complete historical S&P dataset

In [2]:
# calc_sp500_survivors with get_latest=True will:

#   * Dowload the latest metadata from Wikipedia

#   * Assign a complete list of stocks that have ever made up part of
#       the S&P500

#   * Download complete up to date stock price data from Yahoo Finance

#   * Save the complete dataset to memory
#       default filename sp500alltime.csv

#   * Account for survivorship bias in added and removed stocks

#   * Save the final completed working dataset to memory 
#       default filename sp500alltimesurvivors.csv

df_survivors = tad.calc_sp500_survivors(get_latest=True)

[*********************100%***********************]  848 of 848 completed

184 Failed downloads:
['FRE', 'MFE', 'HNG', 'AYE', 'ARG', 'COV', 'CTX', 'JNY', 'CSC', 'KSE', 'WFR', 'PCP', 'CVH', 'PETM', 'SBL', 'TRB', 'ABK', 'SMS', 'BHI', 'ANR', 'SNI', 'OMX', 'ACE', 'HNZ', 'SLR', 'GAS']: YFPricesMissingError('possibly delisted; no price data found  (1d 1900-01-01 -> 2026-04-11)')
['KRFT', 'CMCSK', 'BS', 'MJN', 'DJ', 'NYX', 'MIL', 'BF.B', 'GGP', 'CVC', 'HCBK', 'TYC', 'EK', 'WFM', 'LEH', 'LVLT', 'CEPH', 'KFT', 'DPS', 'SIAL', 'NVLS', 'SWY', 'BCR', 'LO', 'JOY', 'XTO', 'ACAS', 'GMCR', 'JNS', 'FDO', 'TWC', 'CPGX', 'LXK', 'APOL', 'AV', 'QTRN', 'BJS', 'JDSU', 'HSP', 'CFN', 'LDW', 'FNM', 'PGN', 'BXLT', 'WYN', 'NOVL', 'RAI', 'SRCL', 'STJ', 'BRCM', 'LLTC', 'MWW']: YFPricesMissingError('possibly delisted; no price data found  (1d 1900-01-01 -> 2026-04-11) (Yahoo error = "1d data not available for startTime=-2208971040 and endTime=1775880000. Only 100 years worth of day granularity data are allowed to be f

data loaded


In [ ]:
# Dataset is complete from date of inception until the current day
ht(df_survivors)

Price            Close                                                     \
Ticker               A        AA AAL AAP        AAPL        ABBV ABK ABMD   
Date                                                                        
1962-01-02         NaN  1.473408 NaN NaN         NaN         NaN NaN  NaN   
1962-01-03         NaN  1.495946 NaN NaN         NaN         NaN NaN  NaN   
1962-01-04         NaN  1.495946 NaN NaN         NaN         NaN NaN  NaN   
2026-04-08  116.919998       NaN NaN NaN  258.899994  211.589996 NaN  NaN   
2026-04-09  115.389999       NaN NaN NaN  260.489990  212.399994 NaN  NaN   
2026-04-10  115.059998       NaN NaN NaN  260.480011  207.940002 NaN  NaN   

Price                       ...    Volume                                 \
Ticker            ABNB ABS  ...       XRX XTO        XYL        XYZ YHOO   
Date                        ...                                            
1962-01-02         NaN NaN  ...   51233.0 NaN        NaN        NaN  NaN   
1962-01-03         NaN NaN  ...   51233.0 NaN        NaN        NaN  NaN   
1962-01-04         NaN NaN  ...  198099.0 NaN        NaN        NaN  NaN   
2026-04-08  131.399994 NaN  ...       NaN NaN  2933600.0  7033100.0  NaN   
2026-04-09  129.160004 NaN  ...       NaN NaN  2303600.0  5932400.0  NaN   
2026-04-10  128.960007 NaN  ...       NaN NaN  1626900.0  6040300.0  NaN   

Price                                                       
Ticker            YUM        ZBH      ZBRA ZION        ZTS  
Date                                                        
1962-01-02        NaN        NaN       NaN  NaN        NaN  
1962-01-03        NaN        NaN       NaN  NaN        NaN  
1962-01-04        NaN        NaN       NaN  NaN        NaN  
2026-04-08  1213500.0  2128500.0  788900.0  NaN  2635500.0  
2026-04-09  1184400.0  1712400.0  608100.0  NaN  2439100.0  
2026-04-10  1458400.0  1769300.0  552600.0  NaN  2384000.0  

[6 rows x 4240 columns]

### Quality checks still hold true

In [18]:
# Removed
ht(df_survivors.loc[:, pd.IndexSlice[: ,'PENN']].loc[df_removed[df_removed['Ticker']=='PENN'].index[0]:],2)

Price,Close,High,Low,Open,Volume
Ticker,PENN,PENN,PENN,PENN,PENN
Date,,,,,
2022-09-19,31.41,31.67,30.01,30.01,7171400.0
2022-09-20,NaN,NaN,NaN,NaN,NaN
2026-04-09,NaN,NaN,NaN,NaN,NaN
2026-04-10,NaN,NaN,NaN,NaN,NaN


In [19]:
# Added
ht(df_survivors.loc[:, pd.IndexSlice[: ,'ACGL']].loc[:df_added[df_added['Symbol']=='ACGL'].index[0]].tail(),2)

Price,Close,High,Low,Open,Volume
Ticker,ACGL,ACGL,ACGL,ACGL,ACGL
Date,,,,,
2022-10-26,NaN,NaN,NaN,NaN,NaN
2022-10-27,NaN,NaN,NaN,NaN,NaN
2022-10-31,NaN,NaN,NaN,NaN,NaN
2022-11-01,52.717812,55.104569,52.613216,54.695682,9579300.0
